# 03.1 — Apply Business Semantics: Hiring Analytics Metric View

Creates a governed **metric view** (`hiring_analytics`) on the Gold `candidate_scoring_summary` table.  
Metric views expose pre-defined dimensions and measures that Genie and AI/BI Dashboards  
can query without ad-hoc SQL, ensuring consistent, governed business KPIs.

**Dimensions:** JobId, JobTitle, Department, CandidateSegment, Stage, HireDecision, EducationLevel, Certifications, Location, CurrentCompany  
**Measures:** TotalCandidates, HiredCount, PipelineCandidates, HireRate, AvgTotalScore, 6 per-category score averages, AvgCultureFit, AvgYearsHrExperience

The view is attached to the Hiring Analytics Genie space (notebook 04) as a governed metric-view source.

## Setup

Configure catalog/schema and set the target Unity Catalog context.

In [ ]:
import os

_nb_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
project_root = "/Workspace" + _nb_path.rsplit("/notebooks", 1)[0]
try:
    from dotenv import load_dotenv
    load_dotenv(f"{project_root}/.env")
except ImportError:
    pass

dbutils.widgets.text("catalog", "bx4",      "Catalog")
dbutils.widgets.text("schema",  "hrd_2030", "Schema")

# Widget takes priority (job passes it); fall back to .env for interactive runs
catalog = dbutils.widgets.get("catalog") or os.getenv("TARGET_CATALOG", "bx4")
schema  = dbutils.widgets.get("schema")  or os.getenv("TARGET_SCHEMA", "hrd_2030")

print("catalog:", catalog)
print("schema: ", schema)

## Create Metric View

Define `hiring_analytics` as a governed metric view on the Gold table using YAML METRICS syntax. This exposes pre-defined KPIs that Genie and AI/BI Dashboards can query consistently.

In [ ]:
# -----------------------------------------------------------------------
# Create hiring_analytics metric view on the Gold table
# IMPORTANT: CREATE VIEW WITH METRICS requires the current catalog to be set
# version 1.1 + display_name/synonyms need DBR 17.3+ (serverless / current SQL warehouses).
# Synonyms + comments are what Genie uses to map user phrasing onto governed measures.
# -----------------------------------------------------------------------
spark.sql(f"USE CATALOG {catalog}")
spark.sql(f"USE SCHEMA {schema}")

spark.sql(f"""
CREATE OR REPLACE VIEW {catalog}.{schema}.hiring_analytics
WITH METRICS
LANGUAGE YAML
AS $$
version: 1.1
comment: "Governed hiring analytics for Jackson and Jackson HR Digital across 4 open HR roles (JR001-JR004). C001-C010 are historical candidates with a final hire decision; C011-C020 are the active pipeline (hired IS NULL). Use MEASURE() for every measure."
source: {catalog}.{schema}.candidate_scoring_summary
dimensions:
  - name: JobId
    expr: job_id
    comment: "Open role ID: JR001 Director of HR, JR002 VP Talent Acquisition, JR003 Director Comp & Benefits, JR004 Chief People Officer"
    display_name: "Job ID"
    synonyms: ["role id", "requisition", "job req"]
  - name: JobTitle
    expr: job_title
    comment: "Title of the open role the candidate applied for"
    display_name: "Job Title"
    synonyms: ["role", "position", "job"]
  - name: Department
    expr: department
    comment: "Department that owns the open role (Human Resources, Executive Leadership)"
  - name: CandidateSegment
    expr: CASE WHEN hired IS NULL THEN 'Active Pipeline' ELSE 'Historical' END
    comment: "Historical = C001-C010 with a final decision; Active Pipeline = C011-C020 still being evaluated"
    display_name: "Candidate Segment"
    synonyms: ["pipeline vs historical", "candidate type"]
  - name: Stage
    expr: stage
    comment: "Recruiting stage: Hired, Rejected (historical) or Screening, Interview, Final Round, Awaiting Interview (active pipeline)"
    display_name: "Pipeline Stage"
    synonyms: ["status", "recruiting stage"]
  - name: HireDecision
    expr: CASE WHEN hired = 1 THEN 'Hired' WHEN hired = 0 THEN 'Not Hired' ELSE 'In Pipeline' END
    comment: "Final hiring decision; 'In Pipeline' for active candidates without a decision yet"
    display_name: "Hire Decision"
  - name: EducationLevel
    expr: education_level
    comment: "Candidate highest education level: BA, MA, MBA, PhD"
    display_name: "Education Level"
    synonyms: ["degree", "education"]
  - name: Certifications
    expr: certifications
    comment: "HR certifications held by candidate (SPHR, SHRM-SCP, PHR, SHRM-CP)"
  - name: Location
    expr: location
    comment: "Candidate current location"
  - name: CurrentCompany
    expr: current_company
    comment: "Candidate current employer"
    display_name: "Current Company"
    synonyms: ["employer"]
measures:
  - name: TotalCandidates
    expr: COUNT(DISTINCT candidate_id)
    comment: "Number of unique candidates"
    display_name: "Total Candidates"
    synonyms: ["candidate count", "headcount", "number of candidates"]
  - name: HiredCount
    expr: COUNT(DISTINCT CASE WHEN hired = 1 THEN candidate_id END)
    comment: "Number of historical candidates who were hired"
    display_name: "Hired Count"
  - name: PipelineCandidates
    expr: COUNT(DISTINCT CASE WHEN hired IS NULL THEN candidate_id END)
    comment: "Number of active-pipeline candidates still being evaluated"
    display_name: "Pipeline Candidates"
    synonyms: ["active candidates", "open pipeline"]
  - name: HireRate
    expr: ROUND(AVG(hired) * 100, 1)
    comment: "Percent of historical candidates hired (0-100). Active-pipeline rows (hired NULL) are excluded automatically."
    display_name: "Hire Rate %"
    synonyms: ["hiring rate", "conversion rate", "offer rate"]
  - name: AvgTotalScore
    expr: ROUND(AVG(total_score), 1)
    comment: "Average composite hiring score (0-100): mean of the available category scores"
    display_name: "Avg Total Score"
    synonyms: ["average score", "composite score", "overall score"]
  - name: AvgExperienceScore
    expr: ROUND(AVG(experience_score), 1)
    comment: "Average HR experience score (0-100)"
  - name: AvgLeadershipScore
    expr: ROUND(AVG(leadership_score), 1)
    comment: "Average leadership experience score (0-100)"
  - name: AvgEducationScore
    expr: ROUND(AVG(education_score), 1)
    comment: "Average education score (0-100)"
  - name: AvgCertificationScore
    expr: ROUND(AVG(certification_score), 1)
    comment: "Average certification score based on highest credential held (0-100)"
  - name: AvgSkillsMatchScore
    expr: ROUND(AVG(skills_match_score), 1)
    comment: "Average skills-match score (0-100); NULL for pipeline candidates not yet assessed"
  - name: AvgInterviewScore
    expr: ROUND(AVG(interview_score), 1)
    comment: "Average structured interview score (0-100); NULL for pipeline candidates not yet interviewed"
  - name: AvgCultureFit
    expr: ROUND(AVG(culture_fit), 1)
    comment: "Average culture-fit score (0-100); NULL for pipeline candidates not yet interviewed"
  - name: AvgYearsHrExperience
    expr: ROUND(AVG(years_hr_experience), 1)
    comment: "Average years of HR experience"
    display_name: "Avg Years HR Experience"
$$
""")

print(f"Metric view created: {catalog}.{schema}.hiring_analytics")
print("Dimensions: JobId, JobTitle, Department, CandidateSegment, Stage, HireDecision,")
print("            EducationLevel, Certifications, Location, CurrentCompany")
print("Measures:   TotalCandidates, HiredCount, PipelineCandidates, HireRate, AvgTotalScore,")
print("            Avg{Experience,Leadership,Education,Certification,SkillsMatch,Interview}Score,")
print("            AvgCultureFit, AvgYearsHrExperience")


## Verify Metric View

Confirm the metric view is queryable using the `MEASURE()` function, and run sample KPI queries across education level, hire decision, and location.

In [ ]:
# -----------------------------------------------------------------------
# Verify: query the metric view using MEASURE() function (required for metric views)
# -----------------------------------------------------------------------
try:
    result = spark.sql(f"""
        SELECT
            EducationLevel,
            MEASURE(AvgTotalScore)    AS AvgTotalScore,
            MEASURE(TotalCandidates) AS TotalCandidates,
            MEASURE(HireRate)        AS HireRate
        FROM {catalog}.{schema}.hiring_analytics
        GROUP BY EducationLevel
        ORDER BY AvgTotalScore DESC
    """)
    result.show(truncate=False)
    print(f"✓ Metric view '{catalog}.{schema}.hiring_analytics' is queryable.")
except Exception as e:
    print(f"Note: metric view query returned: {e}")
    print("The metric view was created successfully and is available in the Catalog.")

In [ ]:
# -----------------------------------------------------------------------
# Analytics query 1: Hire rate and avg score by education level
# -----------------------------------------------------------------------
try:
    education_kpi = spark.sql(f"""
        SELECT
            EducationLevel,
            MEASURE(TotalCandidates)           AS TotalCandidates,
            ROUND(MEASURE(AvgTotalScore), 1)   AS AvgTotalScore,
            MEASURE(HireRate)                  AS HireRatePct,
            ROUND(MEASURE(AvgLeadershipScore), 1) AS AvgLeadershipScore
        FROM {catalog}.{schema}.hiring_analytics
        GROUP BY EducationLevel
        ORDER BY AvgTotalScore DESC
    """)
    print("KPI: Hire Rate & Avg Score by Education Level")
    education_kpi.show(truncate=False)
except Exception as e:
    print(f"Analytics query note: {e}")

In [ ]:
# -----------------------------------------------------------------------
# Analytics query 2: Score breakdown — hired vs. not hired
# -----------------------------------------------------------------------
try:
    hire_comparison = spark.sql(f"""
        SELECT
            HireDecision,
            MEASURE(TotalCandidates)                AS TotalCandidates,
            ROUND(MEASURE(AvgTotalScore), 1)        AS AvgTotalScore,
            ROUND(MEASURE(AvgExperienceScore), 1)   AS AvgExperienceScore,
            ROUND(MEASURE(AvgLeadershipScore), 1)   AS AvgLeadershipScore,
            ROUND(MEASURE(AvgCertificationScore), 1) AS AvgCertificationScore,
            ROUND(MEASURE(AvgInterviewScore), 1)    AS AvgInterviewScore
        FROM {catalog}.{schema}.hiring_analytics
        GROUP BY HireDecision
        ORDER BY HireDecision
    """)
    print("KPI: Score Breakdown — Hired vs. Not Hired")
    hire_comparison.show(truncate=False)
except Exception as e:
    print(f"Analytics query note: {e}")

In [ ]:
# -----------------------------------------------------------------------
# Analytics query 3: Candidate count and avg score by location
# -----------------------------------------------------------------------
try:
    location_kpi = spark.sql(f"""
        SELECT
            Location,
            MEASURE(TotalCandidates)          AS TotalCandidates,
            ROUND(MEASURE(AvgTotalScore), 1)  AS AvgTotalScore,
            MEASURE(HireRate)                 AS HireRatePct
        FROM {catalog}.{schema}.hiring_analytics
        GROUP BY Location
        ORDER BY TotalCandidates DESC, AvgTotalScore DESC
    """)
    print("KPI: Talent Pool by Location")
    location_kpi.show(truncate=False)
except Exception as e:
    print(f"Analytics query note: {e}")

In [0]:
# -----------------------------------------------------------------------
# Verify metric view is registered in Unity Catalog
# -----------------------------------------------------------------------
views = spark.sql(f"SHOW VIEWS IN {catalog}.{schema}").filter("viewName = 'hiring_analytics'")
views.show(truncate=False)

print("\nMetric view 'hiring_analytics' is registered and queryable.")
print("It can now be added to a Genie Space or AI/BI Dashboard.")

## Summary & Next Steps

### What Was Built

| Asset | Location | Purpose |
|-------|----------|---------|
| `hiring_analytics` metric view | `bx4.hrd_2030` | Governed KPIs for Genie and AI/BI Dashboards |
| 5 dimensions | EducationLevel, Location, Certifications, HireDecision, CurrentCompany | Slice-and-dice axes for analysis |
| 7 measures | AvgTotalScore, HireRate, TotalCandidates, + 4 score averages | Pre-aggregated, semantically named KPIs |

### Why Metric Views Matter

- **Governed consistency**: Every query against `hiring_analytics` uses the same business definitions
- **Genie-ready**: Genie understands the semantic layer without needing raw SQL exploration
- **AI/BI Dashboard compatible**: Drag-and-drop dimensions and measures in dashboard editors
- **Reusable**: One view powers Genie, dashboards, and downstream agents

### Next Steps

1. **`04_create_genie_space`** — Add `candidate_scoring_summary` and `hiring_analytics` to a Genie Space with curated questions
2. **AI/BI Dashboard** — Create visualizations using the metric view dimensions and measures
3. **Agent** — `hire_right_agent.py` queries the Gold table and uses vector search on resumes for compound AI hiring recommendations